# Initialise SageMaker + Upload Data to S3

In [1]:
import sagemaker
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.helper import IamRoleResolver

#1. Initialise session and IAM role
sagemaker_session  = Session()
#role = get_execution_role()
try:
    role = get_execution_role()
except Exception:
    role = boto3.client("sts").get_caller_identity()["Arn"]
#role = IamRoleResolver().create_execution_role(role_type="training")
bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name
prefix = "sagemaker/breast_cancer-logistic"

print(f"Using S3 Bucket: {bucket}")

#Retrieve the official Scikit-Learn training container image
sklearn_training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.2-1",
    py_version="py3",
    image_scope="training"
)
print(f"Training Image: {sklearn_training_image}")

#2. Upload the local breast_cancer_data.csv to S3
train_s3_path = sagemaker_session.upload_data(
    path = "breast_cancer_data.csv",
    bucket = bucket,
    key_prefix = f"{prefix}/data"
)
print(f"Dataset successfully uploaded to: {train_s3_path}")


Using S3 Bucket: sagemaker-eu-north-1-970076636013


NameError: name 'image_uris' is not defined

# Run the Training 

In [18]:

from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, InputData, Compute
from sagemaker.core import image_uris
#1. Set up where the training scirpt is
source_code = SourceCode(
    source_dir=".",
    entry_script = "train.py"
)
#2. Set up the compute instance
compute_config = Compute(
    instance_type = "ml.m5.large",
    instance_count = 1
)

#3. Instantiate the ModelTrainer
model_trainer = ModelTrainer(
    training_image = sklearn_training_image,
    source_code = source_code,
    compute = compute_config,
    role = role,
    sagemaker_session = sagemaker_session,
    base_job_name="sklearn-logistic-breast-cancer",
    hyperparameters={
        "C": 1.0,
        "max-iter": 1000
    }
)

# 4. Start the training (input data as a list)
model_trainer.train(
    input_data_config=[
        InputData(channel_name="train", data_source=train_s3_path)
    ]
)

if hasattr(model_trainer, "_latest_training_job") and model_trainer._latest_training_job:
    model_artifacts_uri = model_trainer._latest_training_job.model_artifacts.s3_model_artifacts
else:
    job_name = model_trainer.latest_training_job_name
    #Output model artifacts path from the completed training
    model_artifacts_uri = f"s3://{bucket}/{prefix}/output/{job_name}/output/model.tar.gz"

print(f"Model artifacts saved at: {model_artifacts_uri}")
    


INFO:sagemaker.core.helper.iam_role_resolver:Role 'arn:aws:iam::970076636013:role/SageMaker-AutoRole-Training' is missing permissions for: ecr:GetAuthorizationToken, cloudwatch:PutMetricData, ec2:CreateNetworkInterface, ec2:CreateNetworkInterfacePermission, ec2:DeleteNetworkInterface...


RoleValidationError: IAM role 'arn:aws:iam::970076636013:role/SageMaker-AutoRole-Training' cannot be used for 'training' workloads.
Missing permissions: cloudwatch:PutMetricData, ec2:CreateNetworkInterface, ec2:CreateNetworkInterfacePermission, ec2:DeleteNetworkInterface, ec2:DeleteNetworkInterfacePermission, ec2:DescribeDhcpOptions, ec2:DescribeNetworkInterfaces, ec2:DescribeSecurityGroups, ec2:DescribeSubnets, ec2:DescribeVpcs, ecr:GetAuthorizationToken

To fix this, either:
  1. Grant the permissions above (and ensure 'sagemaker.amazonaws.com' is a trusted principal) on a role, then pass it via the interface's 'role'/'role_arn' parameter. If you cannot modify or create IAM roles, ask your administrator to do this; or
  2. Create a dedicated least-privilege role and pass its ARN:
       from sagemaker.core.helper import IamRoleResolver
       role_arn = IamRoleResolver().create_execution_role(role_type='training')

# Model Deployment 

In [11]:
from sagemaker.serve import ModelBuilder

#Build + Deploy the model

model_builder = ModelBuilder(
    model_data =model_artifacts_uri,
    role = role,
    sagemaker_session = sagemaker_session,
    image_uri=sklearn_training_image
)

built_model = model_builder.build()

predictor = built_model.deploy(
    initial_instance_count = 1,
    instance_type = "ml.m5.large"
)

print("Endpoint deployed successfully!")

NameError: name 'model_artifacts_uri' is not defined

In [ ]:
predictor.delete_endpoint()